# Install Dependencies

In [ ]:
!pip install -q openai

In [ ]:
!pip install -q sacrebleu
!pip install -q rouge-score
!pip install -q bert-score
!pip install git+https://github.com/huggingface/evaluate@32546aafec25cdc2a5d7dd9f941fc5be56ba122f

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.8/51.8 kB 5.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 104.1/104.1 kB 8.0 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.1/61.1 kB 5.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 4.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 122.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 100.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 61.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 1.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 11.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 44.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 20.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.5/207.5 MB 6.2 MB/s eta 0:00:00

# Load Modules

In [ ]:
import pandas as pd
import json
from openai import OpenAI
from sklearn.model_selection import train_test_split
import re

client = OpenAI(api_key='API_KEY')

# Define Functions

In [ ]:
def format_prompt(data):

    """

    Background: This function helps format data into a list of dicts into the required shape for fine tuning

    Params:
    data (list): list of dicts

    Returns:
    training_data_list (list): a list in the proper format for converting to jsonl

    """

    training_data_list = []

    for x in data:

        updated_data = {
            "messages": [
                {
                    "role": "system",
                    "content": x['system_message']
                },
                {
                    "role": "user",
                    "content": x['user_content']
                }
            ]
        }

        training_data_list.append(updated_data)

    print(training_data_list)

    return training_data_list

In [ ]:
def convert_to_jsonl_and_save(data_list, filename):

    """
    Background:
    This function converts the data_list provided into a jsonl file

    Params:
    data_list (list): a list of a dict ready to convert to jsonl
    filename (str): the name of the filename we want to convert

    """

    with open(filename, 'w') as file:
        for data_dict in data_list:
            json_str = json.dumps(data_dict)  # Convert dictionary to JSON string
            file.write(json_str + '\n')  # Write to file with a newline

    print(f"✅ Data successfully written to {filename}")

In [ ]:
def predict(test, model):
  response = client.chat.completions.create(
      model = model,
      messages= test,
      temperature = 0.2,
      max_tokens= 512
  )
  return response.choices[0].message.content

In [ ]:
def store_predictions(test_df, model, test_data):
  print("fine tuned model id is :", model)
  test_df['Prediction']= None

  for index, row in test_df.iterrows():
    test_message = test_data[index]['messages']
    prediction_result = predict(test_message, model)
    test_df.at[index, 'Prediction'] = prediction_result

  test_df.to_csv("predictions.csv")

# Load Model

In [ ]:
model = 'gpt-4o-2024-08-06'

# Load Data

In [ ]:
data = pd.read_excel('summarization_data.xlsx')

In [ ]:
import re

def normalize_arabic(text):
    text = re.sub(r"[إأآا]", "ا", text)
    text = re.sub(r"ى", "ي", text)
    text = re.sub(r"ؤ", "و", text)
    text = re.sub(r"ئ", "ي", text)
    text = re.sub(r"ة", "ه", text)
    text = re.sub(r"[ًٌٍَُِّْ]", "", text)  # Remove short vowels (diacritics)
    text = re.sub(r"[^\w\s]", "", text)    # Remove punctuation (optional)
    return text.strip()

# Zero Shot

## Prepare Prompt

In [ ]:
data['system_message'] = 'You are provided with a detailed Arabic passage. Your task is to carefully read and comprehend the content, then generate a concise and coherent summary in one or two sentences that accurately captures the main ideas and key points.'
data['user_content'] = 'Passage: \n' + data['text'] + '\n Summary:'

data2 = data
data = data[['system_message','user_content']]

In [ ]:
data.to_excel('SA-ZeroShot-gpt.xlsx')

In [ ]:
# Convert to JSON String
data = data.to_json(orient='records')

# parse JSON string and convert it into a list of python dictionaries
datadict = json.loads(data)

## Format Prompt

In [ ]:
# format dict into shape for fine tuning
zero_shot_data = format_prompt(datadict)

[{'messages': [{'role': 'system', 'content': 'You are provided with a detailed Arabic passage. Your task is to carefully read and comprehend the content, then generate a concise and coherent summary in one or two sentences that accurately captures the main ideas and key points.'}, {'role': 'user', 'content': 'Passage: \nواستحوذت القصة على نقاشات رواد مواقع التواصل الاجتماعي، حيث تصدرت وسوم مثل "#ابن_القاضي وشرطي المرور" قائمة المواضيع المتداولة على تويتر لساعات طويلة. واكب المغردون تطورات القضية بدءا من مرحلة استجواب الطفل والإفراج عنه وحتى إعادة توقيف وإيداعه لدار رعاية في وقت لاحق. فقد أفادت تقارير صحفية بإعادة توقيف الطفل وأربعة من رفاقه صباح الاثنين 2 نوفمبر/ تشرين الثاني. ووصفت تلك التقارير حالة "الهلع والبكاء" التي انتابت الأطفال. للقصة جوانب ودلالات عديدة، فـ "بـطلها طفل يقود سيارة فارهة ويعرض حياته وحياة المارة للخطر، ويهين شرطيا طالبه باستظهار رخصته". مواضيع قد تهمك نهاية كل تلك المعلومات كانت كافية لجعل الموضوع يتصدر اهتمام المصريين، الذين حذروا أيضا مما وصفوها بـ"ظاهرة الإفل

In [ ]:
# Save as jsonl
convert_to_jsonl_and_save(zero_shot_data, 'SA-ZeroShot.jsonl')

✅ Data successfully written to SA-ZeroShot.jsonl


## Predict

In [ ]:
# response = client.chat.completions.create(
#       model = model,
#       messages= zero_shot_data[1]['messages'],
#       temperature = 0.2,
#       max_tokens= 512
#   )

# response.choices[0].message.content

'Neutral'

In [ ]:
true = pd.read_excel('summarization_data.xlsx')
y_true = true['summary'].values

In [ ]:
zs = pd.DataFrame()

zs['text'] = data2['text']
zs['summary'] = y_true
zs = zs.reset_index(drop=True)
zs.head()

,text,summary
0,واستحوذت القصة على نقاشات رواد مواقع التواصل ا...,أثار مقطع فيديو يظهر طفلا يقود سيارة ويعتدي لف...
1,وسيوفر البرنامج خمسة كيلوغرامات من الحبوب الرخ...,دشنت الحكومة الهندية برنامجا ضخما لتوفير الغذا...
2,كان ديفيد سورنسن يعمل كاتب خطابات في البيت الأ...,أصبح ديفيد سورنسن، كاتب الخطابات في البيت الأب...
3,ظهرت أعراض اصابات متوسطة في الدماغ وفقدان سمع ...,تعرض العاملون في السفارة الأمريكية في كوبا إلى...
4,وانطلق نقاش محتدم حول الصورة بعد أن نشرها الأم...,استأثرت صورة امرأة في صفوف الحرس الملكي السعود...


In [ ]:
store_predictions(zs, model, zero_shot_data)

fine tuned model id is : gpt-4o-2024-08-06


In [ ]:
pred_zero = pd.read_csv('TextSummarization-GPT4o-ZeroShot-predictions.csv')

In [ ]:
from rouge_score import rouge_scorer
from bert_score import score as bert_score
import sacrebleu
import evaluate
from transformers import AutoTokenizer

# Initialize storage
bleu_scores = []
bleu_scores2 = []
rougeL_p = []
rougeL_r = []
rougeL_f = []
bertscore_p = []
bertscore_r = []
bertscore_f = []

# Initialize scorer
model_name = 'aubmindlab/bert-base-arabertv2'
tokenizer = AutoTokenizer.from_pretrained(model_name)
rouge = rouge_scorer.RougeScorer(['rougeL'], tokenizer=tokenizer)

# Iterate through rows
for _, row in pred_zero.iterrows():
    raw_references = [row['summary']]  # One reference as list of strings
    raw_predictions = [row['Prediction']]  # One prediction

    # Normalize Arabic
    hyp = [normalize_arabic(text) for text in raw_predictions]
    ref = [[normalize_arabic(ref) for ref in raw_references]]  # Nested list for multiple refs per prediction

    # BLEU
    bleu = sacrebleu.corpus_bleu(hyp, ref)
    bleu_scores.append(bleu.score)

    bleu2 = evaluate.load("bleu")
    results = bleu2.compute(predictions= hyp, references= ref)
    bleu_scores2.append(results['bleu'])

    # ROUGE-L
    r_scores = rouge.score(row['summary'], row['Prediction'])
    rougeL_p.append(r_scores['rougeL'].precision)
    rougeL_r.append(r_scores['rougeL'].recall)
    rougeL_f.append(r_scores['rougeL'].fmeasure)

# BERTScore
P, R, F = bert_score(pred_zero['Prediction'].tolist(), pred_zero['summary'].tolist(), lang="ar", model_type="bert-base-multilingual-cased", verbose=False)
bertscore_p = P.tolist()
bertscore_r = R.tolist()
bertscore_f = F.tolist()

# Create summary DataFrame
metrics_df = pd.DataFrame({
    "BLEU1": bleu_scores,
    "BLEU2": bleu_scores2,
    "ROUGE_L_P": rougeL_p,
    "ROUGE_L_R": rougeL_r,
    "ROUGE_L_F": rougeL_f,
    "BERT_P": bertscore_p,
    "BERT_R": bertscore_r,
    "BERT_F": bertscore_f,
})

# Calculate min, max, mean
summary_stats = metrics_df.agg(['min', 'max', 'mean'])

print(summary_stats)


/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/611 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/384 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

...

config.json:   0%|          | 0.00/625 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/996k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.96M [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/714M [00:00<?, ?B/s]

          BLEU1     BLEU2  ROUGE_L_P  ROUGE_L_R  ROUGE_L_F    BERT_P  \
min    0.000000  0.000000   0.007092   0.027027   0.011236  0.574788   
max   17.548433  0.175484   0.438356   0.633333   0.470588  0.777178   
mean   2.961511  0.009043   0.131799   0.265503   0.171988  0.688352   

        BERT_R    BERT_F  
min   0.591935  0.591594  
max   0.882943  0.799640  
mean  0.730025  0.708294  


# Few Shot

## Prepare Prompt

In [ ]:
data['system_message'] = '''You are provided with a detailed Arabic passage. Your task is to carefully read and comprehend the content, then generate a concise and coherent summary in one or two sentences that accurately captures the main ideas and key points.

Example 1:
Passage:
وكان الرئيس الأوكراني المؤقت، الكسندر تورتشينوف، قد أمر بسحب جميع القوات الأوكرانية من القرم. وسيطرت قوات روسية صباح الاثنين على قاعدة بحرية أوكرانية في فيودوسيا، في ثالث هجوم من نوعه خلال 48 ساعة، وذلك بحسب تصريحات مسؤولين أوكرانيين لبي بي سي . وقال المتحدث باسم وزارة الدفاع الأوكرانية فلاديسلاف سيليزنيوف إن القوات الروسية هاجمت القاعدة وألقت القبض على الجنود الأوكرانيين في قاعدة فيودوسيا وقيدت أيادي ضباطهم. ومن المتوقع أن تسيطر الأزمة الأوكرانية على قمة مجموعة الدول الصناعية السبع في لاهاي. مواضيع قد تهمك نهاية وأكد الرئيس الأمريكي باراك أوباما خلال لقاء مع نظيره الصيني شى جين بينغ على أن \"واشنطن وبكين يمكنهما، بالعمل سويا، تعزيز القانون الدولي واحترام سيادة الدول\". وتسيطر قوات روسية حاليا على معظم القواعد العسكرية الأوكرانية في القرم التي أعلنت موسكو ضمها للاتحاد الروسي بعد استفتاء أجرته السلطات المحلية هناك. قلق بالغ وقال مارك لوين، مراسل بي بي سي في القرم، إن القوات الروسية تسيطر بشكل كامل على القاعدة، ونقلت الجنود الأوكرانيين بعيدا إلى مكان مجهول. وتعد قاعدة فيودوسيا واحدة من آخر القواعد العسكرية التي بقيت تحت سيطرة كييف، لكن قوات روسية ظلت تحاصرها لبعض الوقت، حسبما أفاد مراسلنا. واقتحمت القوات الروسية قاعدتين أخريين وسيطرت عليهما يوم الجمعة. وكان مسؤولون عسكريون روس أعلنوا في وقت سابق أن العلم الروسي أصبح يرفرف على 189 وحدة ومنشأة عسكرية أوكرانية في القرم. وقال ديفيد ستيرن مراسل بي بي سي في كييف إن الأوكرانيين يتابعون هذه التطورات بقلق بالغ. وأشار إلى أن السؤال الذي يدور الآن هو ماذا سيكون رد فعل أوكرانيا والغرب وما هي الخطوة الروسية المقبلة. وحذر قائد الناتو في أوروبا يوم الأحد من أن القوات الروسية المنتشرة على الحدود الشرقية لأوكرانيا قادرة على شن عملية تمتد حتى مولدوفا. قمة الدول الصناعية الكبرى أوباما: العقوبات الغربية على موسكو ستؤثر على الاقتصاد الروسي. ويلقي ضم روسيا لمنطقة القرم بظلاله على قمة مجموعة السبع، التي كان مزمعا عقدها منذ فترة طويلة، بشأن تهديدات الأمن النووي. ومن المتوقع أن يبحث زعماء المجموعة موقفا موحدا حيال الأزمة. وأكد الرئيس الأمريكي على أن أوروبا والولايات المتحدة متفقون على دعم الحكومة الأوكرانية وشعبها، مشيرا إلى أن العقوبات التي فرضت على موسكو ستؤثر على الاقتصاد الروسي. ومن المقرر أن يلتقي وزير الخارجية الأمريكي، جون كيري، مع نظيره الروسي، سيرغي لافروف، على هامش قمة مجموعة السبع. انقطاع الكهرباء من جهة أخرى، شكا سكان محليون في بعض مناطق القرم من انقطاع الكهرباء في وقت متأخر من الأحد. ولف الظلام العديد من المدن من بينها بعض أحياء العاصمة سيمفربول. وقالت شركة توريد الكهرباء في القرم \"كريمنيرغو\" في بيان بث على موقعها الإلكتروني إن عطلا فنيا أصاب أحد الخطوط التي تديرها شركة الكهرباء الوطنية الأوكرانية \"اوكرينيرغو\". ولم يتسن الحصول على تعليق من اوكرينيرغو، ولم يصدر أيضا تأكيد مستقل حول سبب انقطاع الكهرباء. ضم القرم وضمت روسيا القرم إليها عقب استفتاء أجري في المنطقة في 16 مارس/آذار. وجاءت الخطوة الروسية بعد أن أطاحت احتجاجات بالرئيس الأوكراني السابق الموالي لروسيا فيكتور يانوكوفيتش. وأكدت روسيا أنها تحركت لحماية مواطني القرم المتحدرين من أصول روسية ضد من وصفتهم \"بالفاشيين\" الذين انتقلوا إليها من البلد الأم أوكرانيا. وردت الولايات المتحدة والاتحاد الأوروبي بفرض سلسلة من العقوبات ضد أفراد من بينهم مسؤولون بارزون اتهمتهم واشنطن وبروكسل بلعب دور في ضم القرم. موالون لروسيا يتظاهرون في مدينة مدينة دونيتسك، شرقي أوكرانيا. تسيطر قوات روسية حاليا على معظم القواعد العسكرية الأوكرانية في القرم.

Summary:
بدأت القوات الأوكرانية الانسحاب من شبه جزيرة القرم.

Example 2:
Passage:
وذكرت وكالة الأنباء المحلية (جي.إن.إس) أن جماعة \"جيش محمد\" المتشددة أعلنت مسؤوليتها عن الهجوم. لكن ما هي منطقة كشمير المتنازع عليها بين الهند وباكستان؟ خلال العقود الست الماضية ظلت منطقة كشمير القريبة من جبال الهيمالايا محل نزاع بين الهند وباكستان. الجنة الملعونة: دموع الفقراء في كشمير لماذا يلجأ الناس إلى أضرحة الصوفيين في \"كشمير الهندية\"؟ بالصور: الطفولة المسروقة في كشمير فمنذ تقسيم الهند وقيام باكستان عام 1947 وقعت حربان بين البلدين حول منطقة كشمير ذات الأغلبية المسلمة والتي يطالب البلدان بالسيادة عليها. وتعد كشمير اليوم واحدة من أكثر المناطق المدججة بالسلاح في العالم، في ما تدير الصين أجزاء من الإقليم. تسلسل زمني لأهم الأحداث في كشمير

Summary:
قالت الشرطة في القطاع الهندي من إقليم كشمير إن انفجار قنبلة أدى إلى مقتل 40 عنصرا على الأقل من قوات الأمن الخميس، بعد يوم من انفجار أدى لإصابة 12 تلميذا على الأقل.

Example 3:
Passage:
قبل المؤتمر العام الوطني الليبي الاستقالة وقبل المؤتمر العام الوطني (البرلمان الليبي) الاستقالة في جلسة عقدت الأحد. كانت المواجهات وقعت عندما تجمع متظاهرون أمام مقر كتيبة درع ليبيا في المدينة مطالبين بحظر المليشيات. وتحاول الحكومة الليبية منذ سقوط القذافي الحد من نفوذ المليشيات المتزايد، ونزع سلاحها ودمجها بالجيش النظامي. بيد أن الحكومة الليبية التي تواجه صعوبات في تشكيل جيش وشرطة محترفين، تلجأ بانتظام إلى الاعتماد على مليشيات الثوار السابقين لتأمين حدودها أو فض نزاعات قبلية. مواضيع قد تهمك نهاية وفي تشرين الأول/أكتوبر تظاهر سكان بنغازي احتجاجا على المليشيات وطردوا بعضها من قواعدها في المدينة.

Summary:
استقال رئيس الاركان الليبي يوسف المنقوش إثر مقتل 30 شخصا في مواجهات بين عناصر مليشيا مسلحة ومتظاهرين \"مناهضين للمليشيات\" في مدينة بنغازي الليبية.
'''
data['user_content'] = 'The passage you need to summarize\nPassage:\n' + data['text'] +'\nSummary:'

data2 = data
data = data[['system_message','user_content']]

In [ ]:
data.to_excel('SA-FewShot-gpt.xlsx')

In [ ]:
# Convert to JSON String
data = data.to_json(orient='records')

# parse JSON string and convert it into a list of python dictionaries
datadict = json.loads(data)

## Format Prompt

In [ ]:
# format dict into shape for fine tuning
few_shot_data = format_prompt(datadict)

# Save as jsonl
convert_to_jsonl_and_save(few_shot_data, 'SA-FewShot.jsonl')

✅ Data successfully written to SA-FewShot.jsonl


## Predict

In [ ]:
y_true = data2['summary'].values

In [ ]:
fs = pd.DataFrame()

fs['text'] = data2['text']
fs['summary'] = y_true
fs = fs.reset_index(drop=True)
fs.head()

,text,summary
0,واستحوذت القصة على نقاشات رواد مواقع التواصل ا...,أثار مقطع فيديو يظهر طفلا يقود سيارة ويعتدي لف...
1,وسيوفر البرنامج خمسة كيلوغرامات من الحبوب الرخ...,دشنت الحكومة الهندية برنامجا ضخما لتوفير الغذا...
2,كان ديفيد سورنسن يعمل كاتب خطابات في البيت الأ...,أصبح ديفيد سورنسن، كاتب الخطابات في البيت الأب...
3,ظهرت أعراض اصابات متوسطة في الدماغ وفقدان سمع ...,تعرض العاملون في السفارة الأمريكية في كوبا إلى...
4,وانطلق نقاش محتدم حول الصورة بعد أن نشرها الأم...,استأثرت صورة امرأة في صفوف الحرس الملكي السعود...


In [ ]:
store_predictions(fs, model, few_shot_data)

fine tuned model id is : gpt-4o-2024-08-06


In [ ]:
pred_few = pd.read_csv('TextSummarization-GPT4o-FewShot-predictions.csv')

In [ ]:
from rouge_score import rouge_scorer
from bert_score import score as bert_score
import sacrebleu
import evaluate
from transformers import AutoTokenizer

# Initialize storage
bleu_scores = []
bleu_scores2 = []
rougeL_p = []
rougeL_r = []
rougeL_f = []
bertscore_p = []
bertscore_r = []
bertscore_f = []

# Initialize scorer
model_name = 'aubmindlab/bert-base-arabertv2'
tokenizer = AutoTokenizer.from_pretrained(model_name)
rouge = rouge_scorer.RougeScorer(['rougeL'], tokenizer=tokenizer)

# Iterate through rows
for _, row in pred_few.iterrows():
    raw_references = [row['summary']]  # One reference as list of strings
    raw_predictions = [row['Prediction']]  # One prediction

    # Normalize Arabic
    hyp = [normalize_arabic(text) for text in raw_predictions]
    ref = [[normalize_arabic(ref) for ref in raw_references]]  # Nested list for multiple refs per prediction

    # BLEU
    bleu = sacrebleu.corpus_bleu(hyp, ref)
    bleu_scores.append(bleu.score)

    bleu2 = evaluate.load("bleu")
    results = bleu2.compute(predictions= hyp, references= ref)
    bleu_scores2.append(results['bleu'])

    # ROUGE-L
    r_scores = rouge.score(row['summary'], row['Prediction'])
    rougeL_p.append(r_scores['rougeL'].precision)
    rougeL_r.append(r_scores['rougeL'].recall)
    rougeL_f.append(r_scores['rougeL'].fmeasure)

# BERTScore
P, R, F = bert_score(pred_few['Prediction'].tolist(), pred_few['summary'].tolist(), lang="ar", model_type="bert-base-multilingual-cased", verbose=False)
bertscore_p = P.tolist()
bertscore_r = R.tolist()
bertscore_f = F.tolist()

# Create summary DataFrame
metrics_df = pd.DataFrame({
    "BLEU1": bleu_scores,
    "BLEU2": bleu_scores2,
    "ROUGE_L_P": rougeL_p,
    "ROUGE_L_R": rougeL_r,
    "ROUGE_L_F": rougeL_f,
    "BERT_P": bertscore_p,
    "BERT_R": bertscore_r,
    "BERT_F": bertscore_f,
})

# Calculate min, max, mean
summary_stats = metrics_df.agg(['min', 'max', 'mean'])

print(summary_stats)


/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


          BLEU1     BLEU2  ROUGE_L_P  ROUGE_L_R  ROUGE_L_F    BERT_P  \
min    0.000000  0.000000   0.032258   0.095238   0.048193  0.561883   
max   24.037659  0.240377   0.431034   0.589744   0.450980  0.789234   
mean   3.506085  0.011535   0.154466   0.257180   0.188269  0.698273   

        BERT_R    BERT_F  
min   0.628667  0.593402  
max   0.858650  0.820413  
mean  0.731463  0.714170  


# CoT

## Prepare Prompt

In [ ]:
data['system_message'] = '''You are provided with a detailed Arabic passage. Your task is to carefully read and comprehend the content, then generate a concise and coherent summary in one or two sentences that accurately captures the main ideas and key points.

Step 1: Comprehend the Passage
Thoroughly read the Arabic text to understand its main theme, event, or subject matter. Focus on grasping the overall context and intent of the passage.

Step 2: Extract Key Information
Identify the most important facts, ideas, or arguments presented. Disregard minor or repetitive details and focus on what’s essential to the overall message.

Step 3: Generate the Summary
Using the key points identified, write a clear and concise summary in one or two sentences. Ensure your summary is coherent, reflects the passage accurately, and avoids adding any personal interpretation.

Example 1:
Passage:
وكان الرئيس الأوكراني المؤقت، الكسندر تورتشينوف، قد أمر بسحب جميع القوات الأوكرانية من القرم. وسيطرت قوات روسية صباح الاثنين على قاعدة بحرية أوكرانية في فيودوسيا، في ثالث هجوم من نوعه خلال 48 ساعة، وذلك بحسب تصريحات مسؤولين أوكرانيين لبي بي سي . وقال المتحدث باسم وزارة الدفاع الأوكرانية فلاديسلاف سيليزنيوف إن القوات الروسية هاجمت القاعدة وألقت القبض على الجنود الأوكرانيين في قاعدة فيودوسيا وقيدت أيادي ضباطهم. ومن المتوقع أن تسيطر الأزمة الأوكرانية على قمة مجموعة الدول الصناعية السبع في لاهاي. مواضيع قد تهمك نهاية وأكد الرئيس الأمريكي باراك أوباما خلال لقاء مع نظيره الصيني شى جين بينغ على أن \"واشنطن وبكين يمكنهما، بالعمل سويا، تعزيز القانون الدولي واحترام سيادة الدول\". وتسيطر قوات روسية حاليا على معظم القواعد العسكرية الأوكرانية في القرم التي أعلنت موسكو ضمها للاتحاد الروسي بعد استفتاء أجرته السلطات المحلية هناك. قلق بالغ وقال مارك لوين، مراسل بي بي سي في القرم، إن القوات الروسية تسيطر بشكل كامل على القاعدة، ونقلت الجنود الأوكرانيين بعيدا إلى مكان مجهول. وتعد قاعدة فيودوسيا واحدة من آخر القواعد العسكرية التي بقيت تحت سيطرة كييف، لكن قوات روسية ظلت تحاصرها لبعض الوقت، حسبما أفاد مراسلنا. واقتحمت القوات الروسية قاعدتين أخريين وسيطرت عليهما يوم الجمعة. وكان مسؤولون عسكريون روس أعلنوا في وقت سابق أن العلم الروسي أصبح يرفرف على 189 وحدة ومنشأة عسكرية أوكرانية في القرم. وقال ديفيد ستيرن مراسل بي بي سي في كييف إن الأوكرانيين يتابعون هذه التطورات بقلق بالغ. وأشار إلى أن السؤال الذي يدور الآن هو ماذا سيكون رد فعل أوكرانيا والغرب وما هي الخطوة الروسية المقبلة. وحذر قائد الناتو في أوروبا يوم الأحد من أن القوات الروسية المنتشرة على الحدود الشرقية لأوكرانيا قادرة على شن عملية تمتد حتى مولدوفا. قمة الدول الصناعية الكبرى أوباما: العقوبات الغربية على موسكو ستؤثر على الاقتصاد الروسي. ويلقي ضم روسيا لمنطقة القرم بظلاله على قمة مجموعة السبع، التي كان مزمعا عقدها منذ فترة طويلة، بشأن تهديدات الأمن النووي. ومن المتوقع أن يبحث زعماء المجموعة موقفا موحدا حيال الأزمة. وأكد الرئيس الأمريكي على أن أوروبا والولايات المتحدة متفقون على دعم الحكومة الأوكرانية وشعبها، مشيرا إلى أن العقوبات التي فرضت على موسكو ستؤثر على الاقتصاد الروسي. ومن المقرر أن يلتقي وزير الخارجية الأمريكي، جون كيري، مع نظيره الروسي، سيرغي لافروف، على هامش قمة مجموعة السبع. انقطاع الكهرباء من جهة أخرى، شكا سكان محليون في بعض مناطق القرم من انقطاع الكهرباء في وقت متأخر من الأحد. ولف الظلام العديد من المدن من بينها بعض أحياء العاصمة سيمفربول. وقالت شركة توريد الكهرباء في القرم \"كريمنيرغو\" في بيان بث على موقعها الإلكتروني إن عطلا فنيا أصاب أحد الخطوط التي تديرها شركة الكهرباء الوطنية الأوكرانية \"اوكرينيرغو\". ولم يتسن الحصول على تعليق من اوكرينيرغو، ولم يصدر أيضا تأكيد مستقل حول سبب انقطاع الكهرباء. ضم القرم وضمت روسيا القرم إليها عقب استفتاء أجري في المنطقة في 16 مارس/آذار. وجاءت الخطوة الروسية بعد أن أطاحت احتجاجات بالرئيس الأوكراني السابق الموالي لروسيا فيكتور يانوكوفيتش. وأكدت روسيا أنها تحركت لحماية مواطني القرم المتحدرين من أصول روسية ضد من وصفتهم \"بالفاشيين\" الذين انتقلوا إليها من البلد الأم أوكرانيا. وردت الولايات المتحدة والاتحاد الأوروبي بفرض سلسلة من العقوبات ضد أفراد من بينهم مسؤولون بارزون اتهمتهم واشنطن وبروكسل بلعب دور في ضم القرم. موالون لروسيا يتظاهرون في مدينة مدينة دونيتسك، شرقي أوكرانيا. تسيطر قوات روسية حاليا على معظم القواعد العسكرية الأوكرانية في القرم.

Thoughts:
- Main Content: The passage discusses the ongoing crisis between Ukraine and Russia following Russia's annexation of Crimea in March 2014. It details military developments, international diplomatic responses, and the humanitarian situation in Crimea, including power outages.
- Key Information: The Ukrainian interim president ordered the withdrawal of troops from Crimea. Russian forces seized a Ukrainian naval base in Feodosia—third such seizure in 48 hours. The U.S. and allies are discussing unified responses at the G7 summit. NATO warned of a possible Russian advance into Moldova. Crimea is now largely controlled by Russian military forces after a disputed referendum. There are civilian hardships in Crimea, including widespread power outages. The annexation has led to Western sanctions on Russia. Political unrest continues in eastern Ukraine (e.g., Donetsk protests).

Summary:
بدأت القوات الأوكرانية الانسحاب من شبه جزيرة القرم.

Example 2:
Passage:
وذكرت وكالة الأنباء المحلية (جي.إن.إس) أن جماعة \"جيش محمد\" المتشددة أعلنت مسؤوليتها عن الهجوم. لكن ما هي منطقة كشمير المتنازع عليها بين الهند وباكستان؟ خلال العقود الست الماضية ظلت منطقة كشمير القريبة من جبال الهيمالايا محل نزاع بين الهند وباكستان. الجنة الملعونة: دموع الفقراء في كشمير لماذا يلجأ الناس إلى أضرحة الصوفيين في \"كشمير الهندية\"؟ بالصور: الطفولة المسروقة في كشمير فمنذ تقسيم الهند وقيام باكستان عام 1947 وقعت حربان بين البلدين حول منطقة كشمير ذات الأغلبية المسلمة والتي يطالب البلدان بالسيادة عليها. وتعد كشمير اليوم واحدة من أكثر المناطق المدججة بالسلاح في العالم، في ما تدير الصين أجزاء من الإقليم. تسلسل زمني لأهم الأحداث في كشمير

Thoughts:
- Main Content: The passage discusses a recent attack claimed by the militant group Jaish-e-Mohammed and provides background on the long-standing dispute over the Kashmir region, which has been a source of conflict between India and Pakistan since 1947.
- Key Information: The militant group Jaish-e-Mohammed claimed responsibility for a recent attack. Kashmir is a historically disputed region between India and Pakistan since 1947. Two wars have been fought over it, and the region has a Muslim majority population. China also controls part of the territory. Kashmir is now one of the most heavily militarized regions in the world. The passage references emotional, religious, and humanitarian issues in the area (e.g., poverty, shrine visits, stolen childhoods).

Summary:
قالت الشرطة في القطاع الهندي من إقليم كشمير إن انفجار قنبلة أدى إلى مقتل 40 عنصرا على الأقل من قوات الأمن الخميس، بعد يوم من انفجار أدى لإصابة 12 تلميذا على الأقل.

Example 3:
Passage:
قبل المؤتمر العام الوطني الليبي الاستقالة وقبل المؤتمر العام الوطني (البرلمان الليبي) الاستقالة في جلسة عقدت الأحد. كانت المواجهات وقعت عندما تجمع متظاهرون أمام مقر كتيبة درع ليبيا في المدينة مطالبين بحظر المليشيات. وتحاول الحكومة الليبية منذ سقوط القذافي الحد من نفوذ المليشيات المتزايد، ونزع سلاحها ودمجها بالجيش النظامي. بيد أن الحكومة الليبية التي تواجه صعوبات في تشكيل جيش وشرطة محترفين، تلجأ بانتظام إلى الاعتماد على مليشيات الثوار السابقين لتأمين حدودها أو فض نزاعات قبلية. مواضيع قد تهمك نهاية وفي تشرين الأول/أكتوبر تظاهر سكان بنغازي احتجاجا على المليشيات وطردوا بعضها من قواعدها في المدينة.

Thoughts:
- Main Content: The passage outlines the complex security situation in post-Gaddafi Libya, particularly in Benghazi, highlighting public protests against militias, the role of former rebel groups, and the government's challenges in establishing professional security forces.
- Key Information: The Libyan General National Congress accepted a resignation in a session held on Sunday. Protests erupted in Benghazi demanding the dissolution of militias, particularly targeting the Libya Shield Brigade. Since the fall of Gaddafi, the government has struggled to disarm militias and integrate them into the regular army. Due to weak formal security institutions, the government often relies on these same militias for tasks like border security and conflict resolution. In October, Benghazi residents protested and expelled some militias from their bases.

Summary:
استقال رئيس الاركان الليبي يوسف المنقوش إثر مقتل 30 شخصا في مواجهات بين عناصر مليشيا مسلحة ومتظاهرين \"مناهضين للمليشيات\" في مدينة بنغازي الليبية.
'''
data['user_content'] = 'The passage you need to summarize\nPassage:\n' + data['text']

data2 = data
data = data[['system_message','user_content']]

In [ ]:
data.to_excel('SA-CoT-gpt.xlsx')

In [ ]:
# Convert to JSON String
data = data.to_json(orient='records')

# parse JSON string and convert it into a list of python dictionaries
datadict = json.loads(data)

## Format Prompt

In [ ]:
# format dict into shape for fine tuning
CoT_data = format_prompt(datadict)

# Save as jsonl
convert_to_jsonl_and_save(CoT_data, 'SA-CoT.jsonl')

✅ Data successfully written to SA-CoT.jsonl


## Predict

In [ ]:
def predict(test, model):
  response = client.chat.completions.create(
      model = model,
      messages= test,
      temperature = 0.2,
      max_tokens= 512
  )
  return response.choices[0].message.content

In [ ]:
def store_predictions(test_df, model, test_data):
  print("fine tuned model id is :", model)
  test_df['Prediction']= None

  for index, row in test_df.iterrows():
    test_message = test_data[index]['messages']
    prediction_result = predict(test_message, model)
    test_df.at[index, 'Prediction'] = prediction_result

  test_df.to_csv("Text Summarization-GPT4o-CoT-predictions.csv")

In [ ]:
y_true = data2['summary'].values

In [ ]:
cot = pd.DataFrame()

cot['text'] = data2['text']
cot['summary'] = y_true
cot = cot.reset_index(drop=True)
cot.head()

,text,summary
0,واستحوذت القصة على نقاشات رواد مواقع التواصل ا...,أثار مقطع فيديو يظهر طفلا يقود سيارة ويعتدي لف...
1,وسيوفر البرنامج خمسة كيلوغرامات من الحبوب الرخ...,دشنت الحكومة الهندية برنامجا ضخما لتوفير الغذا...
2,كان ديفيد سورنسن يعمل كاتب خطابات في البيت الأ...,أصبح ديفيد سورنسن، كاتب الخطابات في البيت الأب...
3,ظهرت أعراض اصابات متوسطة في الدماغ وفقدان سمع ...,تعرض العاملون في السفارة الأمريكية في كوبا إلى...
4,وانطلق نقاش محتدم حول الصورة بعد أن نشرها الأم...,استأثرت صورة امرأة في صفوف الحرس الملكي السعود...


In [ ]:
store_predictions(cot, model, CoT_data)

fine tuned model id is : gpt-4o-2024-08-06


In [ ]:
pred_cot = pd.read_csv('TextSummarization-GPT4o-CoT-predictions.csv')

In [ ]:
from rouge_score import rouge_scorer
from bert_score import score as bert_score
import sacrebleu
import evaluate
from transformers import AutoTokenizer

# Initialize storage
bleu_scores = []
bleu_scores2 = []
rougeL_p = []
rougeL_r = []
rougeL_f = []
bertscore_p = []
bertscore_r = []
bertscore_f = []

# Initialize scorer
model_name = 'aubmindlab/bert-base-arabertv2'
tokenizer = AutoTokenizer.from_pretrained(model_name)
rouge = rouge_scorer.RougeScorer(['rougeL'], tokenizer=tokenizer)

# Iterate through rows
for _, row in pred_cot.iterrows():
    raw_references = [row['summary']]  # One reference as list of strings
    raw_predictions = [row['Prediction']]  # One prediction

    # Normalize Arabic
    hyp = [normalize_arabic(text) for text in raw_predictions]
    ref = [[normalize_arabic(ref) for ref in raw_references]]  # Nested list for multiple refs per prediction

    # BLEU
    bleu = sacrebleu.corpus_bleu(hyp, ref)
    bleu_scores.append(bleu.score)

    bleu2 = evaluate.load("bleu")
    results = bleu2.compute(predictions= hyp, references= ref)
    bleu_scores2.append(results['bleu'])

    # ROUGE-L
    r_scores = rouge.score(row['summary'], row['Prediction'])
    rougeL_p.append(r_scores['rougeL'].precision)
    rougeL_r.append(r_scores['rougeL'].recall)
    rougeL_f.append(r_scores['rougeL'].fmeasure)

# BERTScore
P, R, F = bert_score(pred_cot['Prediction'].tolist(), pred_cot['summary'].tolist(), lang="ar", model_type="bert-base-multilingual-cased", verbose=False)
bertscore_p = P.tolist()
bertscore_r = R.tolist()
bertscore_f = F.tolist()

# Create summary DataFrame
metrics_df = pd.DataFrame({
    "BLEU1": bleu_scores,
    "BLEU2": bleu_scores2,
    "ROUGE_L_P": rougeL_p,
    "ROUGE_L_R": rougeL_r,
    "ROUGE_L_F": rougeL_f,
    "BERT_P": bertscore_p,
    "BERT_R": bertscore_r,
    "BERT_F": bertscore_f,
})

# Calculate min, max, mean
summary_stats = metrics_df.agg(['min', 'max', 'mean'])

print(summary_stats)


          BLEU1     BLEU2  ROUGE_L_P  ROUGE_L_R  ROUGE_L_F    BERT_P  \
min    0.000000  0.000000   0.000000   0.000000   0.000000  0.528827   
max   25.474299  0.254743   0.360000   0.600000   0.446602  0.787956   
mean   2.657251  0.010084   0.105898   0.252551   0.136483  0.655012   

        BERT_R    BERT_F  
min   0.553222  0.567216  
max   0.860556  0.814876  
mean  0.709921  0.680484  
